In [ ]:
# Runtime setup for the Chapter 12 capstone listings.
# The book snippets assume X, y, and seed already exist.
from sklearn.datasets import make_classification

seed = 17
X, y = make_classification(
    n_samples=180,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=seed,
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=seed
)

preprocess = Pipeline([
    ("scale", StandardScaler()),
])

X_train_p = preprocess.fit_transform(X_train)
X_val_p = preprocess.transform(X_val)
X_test_p = preprocess.transform(X_test)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

models = {
    "majority": DummyClassifier(strategy="most_frequent"),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "rbf_svc": GridSearchCV(
        make_pipeline(StandardScaler(), SVC(kernel="rbf")),
        param_grid={"svc__C": [0.1, 1.0, 10.0], "svc__gamma": [0.1, 1.0, 10.0]},
        scoring="balanced_accuracy",
        cv=3,
    ),
}

scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    scores[name] = balanced_accuracy_score(y_val, pred)

In [ ]:
class QuantumClassifier:
    def __init__(self, n_qubits, depth, shots, seed):
        self.n_qubits = n_qubits
        self.depth = depth
        self.shots = shots
        self.seed = seed

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        # Build circuits, optimize parameters, and store diagnostics.
        return self

    def predict_proba(self, X):
        # Return probability-like scores with shape (n_examples, 2).
        raise NotImplementedError

    def resource_record(self):
        # Return shots, circuit count, depth, parameters, and timing.
        raise NotImplementedError

In [ ]:
# Support objects used by the framework-neutral evaluation-loop listing.
from dataclasses import dataclass

import numpy as np
from sklearn.base import clone
from sklearn.preprocessing import FunctionTransformer

@dataclass
class Split:
    X_train: np.ndarray
    X_val: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray

@dataclass
class PreparedData(Split):
    preprocess: Pipeline

def make_split(X, y, seed):
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=seed
    )
    return Split(X_train, X_val, X_test, y_train, y_val, y_test)

def fit_preprocess_on_train(split):
    preprocess = Pipeline([("scale", StandardScaler())])
    return PreparedData(
        X_train=preprocess.fit_transform(split.X_train),
        X_val=preprocess.transform(split.X_val),
        X_test=preprocess.transform(split.X_test),
        y_train=split.y_train,
        y_val=split.y_val,
        y_test=split.y_test,
        preprocess=preprocess,
    )

class CandidateAdapter:
    def __init__(self, name, estimator):
        self.name = name
        self.estimator = estimator
        self.validation_metrics = {}

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        self.estimator.fit(X_train, y_train)
        if X_val is not None and y_val is not None:
            self.validation_metrics = evaluate(self, X_val, y_val)
        return self

    def predict(self, X):
        return self.estimator.predict(X)

    def predict_proba(self, X):
        if hasattr(self.estimator, "predict_proba"):
            return self.estimator.predict_proba(X)
        labels = self.predict(X)
        return np.column_stack([1.0 - labels, labels])

    def resource_record(self):
        return {"kind": "classical", "estimator": type(self.estimator).__name__}

def trig_features(X):
    return np.concatenate([np.sin(X), np.cos(X)], axis=1)

class TrigQuantumSurrogate(QuantumClassifier):
    name = "angle_trig_surrogate"

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        self.estimator = make_pipeline(
            FunctionTransformer(trig_features, validate=False),
            LogisticRegression(max_iter=2000, random_state=self.seed),
        )
        self.estimator.fit(X_train, y_train)
        self.n_features_ = X_train.shape[1]
        if X_val is not None and y_val is not None:
            self.validation_metrics = evaluate(self, X_val, y_val)
        return self

    def predict(self, X):
        return self.estimator.predict(X)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

    def resource_record(self):
        n_features = getattr(self, "n_features_", self.n_qubits)
        return {
            "kind": "quantum-surrogate",
            "shots": self.shots,
            "depth": self.depth,
            "parameters": 2 * n_features * self.depth,
            "circuit_count": self.shots * n_features,
        }

def build_classical_baselines():
    return [
        CandidateAdapter("majority", DummyClassifier(strategy="most_frequent")),
        CandidateAdapter("logistic", LogisticRegression(max_iter=2000, random_state=0)),
        CandidateAdapter("rbf_svc", SVC(C=1.0, gamma=0.1, kernel="rbf", probability=True)),
    ]

def build_quantum_candidates(seed):
    return [TrigQuantumSurrogate(n_qubits=X.shape[1], depth=1, shots=512, seed=seed)]

def evaluate(candidate, X_eval, y_eval):
    pred = candidate.predict(X_eval)
    return {"balanced_accuracy": balanced_accuracy_score(y_eval, pred)}

def select_by_validation_metric(candidates, primary="balanced_accuracy"):
    return max(candidates, key=lambda candidate: candidate.validation_metrics[primary])

seeds = [11, 17, 23]

In [ ]:
records = []

for seed in seeds:
    split = make_split(X, y, seed=seed)
    prepared = fit_preprocess_on_train(split)

    candidates = build_classical_baselines() + build_quantum_candidates(seed)
    for candidate in candidates:
        candidate.fit(prepared.X_train, prepared.y_train,
                      prepared.X_val, prepared.y_val)
        val_metrics = evaluate(candidate, prepared.X_val, prepared.y_val)

    selected = select_by_validation_metric(candidates, primary="balanced_accuracy")
    test_metrics = evaluate(selected, prepared.X_test, prepared.y_test)
    records.append({
        "seed": seed,
        "model": selected.name,
        "validation": selected.validation_metrics,
        "test": test_metrics,
        "resources": selected.resource_record(),
    })

In [ ]:
# Compact run summary for the executed notebook.
for record in records:
    print(
        f"seed={record['seed']} selected={record['model']} "
        f"val={record['validation']['balanced_accuracy']:.3f} "
        f"test={record['test']['balanced_accuracy']:.3f}"
    )